# Single-Core Query Analysis: QueryA vs QueryB RAPL Comparison

This notebook analyzes single-core query timing results with grouped comparisons between queryA and queryB variants.
Data includes per-run RAPL energy measurements and query execution times.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
import re
import glob

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

RUNID_FILTERS = ["E940BCCA0B1AE2A3", "7C61BAE5E86FAB53", "352FE3245A617222", "1F830D594D6929BA"]

# Load and filter the run summary data
runs_all_df = pd.read_csv('../runner/logs/query_timing_single_core.csv')
runs_all_df['run_id'] = runs_all_df['run_id'].astype(str)
available_run_ids = sorted(runs_all_df['run_id'].dropna().unique())
runs_df = runs_all_df[runs_all_df['run_id'].isin(RUNID_FILTERS)].copy()

# Load and filter the perf summary metadata
perf_runs_all_df = pd.read_csv('../runner/logs/single_core_perf_runs.csv')
perf_runs_all_df['run_id'] = perf_runs_all_df['run_id'].astype(str)
perf_runs_df = perf_runs_all_df[perf_runs_all_df['run_id'].isin(RUNID_FILTERS)].copy()

print(f'Run ID filters: {RUNID_FILTERS}')
print(f'Available run IDs: {available_run_ids}')
print(f'Loaded {len(runs_df)} filtered runs from query timing data')
print(f'Loaded {len(perf_runs_df)} filtered rows from perf metadata')
print(f'Columns: {list(runs_df.columns)}')
print(f'\nShape: {runs_df.shape}')
print(f'\nFirst few rows:')
display(runs_df.head())
print(f'\nData types:')
print(runs_df.dtypes)

## Load and Inspect Per-Run RAPL Samples

In [ ]:
# List available sample files
sample_dir = Path('../runner/logs/single_core_logs/single_core_rapl_samples')
sample_files_all = sorted(sample_dir.glob('*.rapl.csv')) if sample_dir.exists() else []

sample_files = [
    path for path in sample_files_all
    if path.name.split('__', 1)[0] in RUNID_FILTERS
]

if sample_files:
    print(f'Using RUNID-filtered sample files for {RUNID_FILTERS}.')
elif sample_files_all:
    sample_files = sample_files_all
    print('No sample files matched RUNID_FILTERS; using all available sample files for the RAPL sections.')
else:
    print(f'No RAPL sample files found under: {sample_dir}')

print(f'Found {len(sample_files)} RAPL sample files')
if sample_files:
    print(f'\nSample file names (first 5):')
    for f in sample_files[:5]:
        print(f'  {f.name}')

In [ ]:
def load_sample(filepath):
    """
    Load a RAPL CSV file and pivot per-metric rows into wide format columns.
    Returns a dataframe with timestamp_sec and energy columns (pkg, cores, gpu, ram).
    """
    df = pd.read_csv(filepath, header=None, comment='#')
    df.columns = ['timestamp_sec', 'value', 'unit', 'metric', 'counter', 'pct', 'col6', 'col7']

    # Pivot: each metric becomes a column
    # Extract metric short names
    df['metric_short'] = df['metric'].str.extract(r'energy-([a-z]+)')[0]

    # For each timestamp, we have multiple metrics; pivot to wide format
    pivoted = df.pivot_table(
        index='timestamp_sec',
        columns='metric_short',
        values='value',
        aggfunc='first'  # In case of duplicates, take first
    )
    pivoted.reset_index(inplace=True)
    pivoted.columns.name = None  # Remove the columns index name

    # Rename columns for clarity
    rename_map = {'cores': 'cores_j', 'dram': 'dram_j', 'gpu': 'gpu_j', 'pkg': 'pkg_j'}
    pivoted.rename(columns=rename_map, inplace=True)

    return pivoted

# Test on a sample file
if not sample_files:
    print('Skipping sample load test because sample_files is empty.')
    sample_test = pd.DataFrame()
else:
    sample_test = load_sample(sample_files[0])
    print(f'Loaded sample with shape {sample_test.shape}')
    print(f'Columns: {list(sample_test.columns)}')
    print(f'\nFirst few rows:')
    display(sample_test.head())

## Full-Loop Window Comparison (1-Second Binned Energy)

In [ ]:
# Two separate plots per base query:
# 1) Package power (1s bins) with queryA/queryB overlaid on same plot
# 2) Turbostat metrics with all CPUs on one plot

from matplotlib.lines import Line2D

# Turbostat parsing adapted from detailed_perf_analysis (whitespace-separated)
def parse_turbostat_file(path):
    """Parse turbostat CSV with whitespace separator, filter headers, extract data grouped by CPU."""
    try:
        df = pd.read_csv(path, sep=r"\s+", engine="python")
    except Exception as e:
        print(f"Failed to parse {path}: {e}")
        return {}
    
    # Filter out header rows where Core column repeats "Core"
    if "Core" in df.columns:
        df = df[df["Core"].astype(str) != "Core"].copy()
    
    # Convert numeric columns
    num_cols = ["PkgWatt", "CorWatt", "RAMWatt", "Avg_MHz", "Bzy_MHz", "Busy%", "IPC"]
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    
    # Get CPU column and convert to numeric
    if "CPU" not in df.columns:
        return {}
    
    df["CPU_num"] = pd.to_numeric(df["CPU"], errors="coerce")
    df = df.dropna(subset=["CPU_num"])
    
    # Group by CPU number and return dict
    cpu_groups = {}
    for cpu_num in sorted(df["CPU_num"].unique()):
        cpu_data = df[df["CPU_num"] == cpu_num].copy()
        if not cpu_data.empty:
            cpu_groups[int(cpu_num)] = cpu_data
    
    return cpu_groups


# Find all turbostat files in the perf runs directory
turbostat_dir = Path('../runner/logs/single_core_logs/single_core_perf_runs')
turbostat_files_all = sorted(turbostat_dir.glob('*.turbostat.csv')) if turbostat_dir.exists() else []

# Filter by run ID
turbostat_files = [f for f in turbostat_files_all if f.name.split('__')[0] in RUNID_FILTERS]

if not turbostat_files:
    print(f"No turbostat files found in {turbostat_dir} matching run IDs {RUNID_FILTERS}")
else:
    print(f"Found {len(turbostat_files)} turbostat files")

# Build mapping: base_query -> run_id -> variant -> {rapl:[], turb:[]}
base_query_groups = {}

for rapl_file in sample_files:
    match = re.match(r"([A-F0-9]+)__(.+)\.rapl\.csv", rapl_file.name)
    if not match:
        continue
    run_id, query_spec = match.groups()

    query_spec_clean = re.sub(r'__loop\d+_run\d+$', '', query_spec)

    if '-queryA' in query_spec_clean:
        base_name = query_spec_clean.replace('-queryA', '').replace('.sql', '')
        variant = 'queryA'
    elif '-queryB' in query_spec_clean:
        base_name = query_spec_clean.replace('-queryB', '').replace('.sql', '')
        variant = 'queryB'
    else:
        continue

    turb_file_name = re.sub(r'__loop\d+_run\d+\.rapl\.csv$', '.turbostat.csv', rapl_file.name)
    turb_file = turbostat_dir / turb_file_name

    base_query_groups.setdefault(base_name, {}) \
        .setdefault(run_id, {}) \
        .setdefault(variant, {'rapl': [], 'turb': []})

    base_query_groups[base_name][run_id][variant]['rapl'].append(rapl_file)
    if turb_file.exists():
        base_query_groups[base_name][run_id][variant]['turb'].append(turb_file)

print(f"\nGrouped into {len(base_query_groups)} base queries")
for q, run_groups in base_query_groups.items():
    print(f"  {q}: runs={list(run_groups.keys())}")

# Plot per base query
for base_query in sorted(base_query_groups.keys()):
    run_ids = sorted(base_query_groups[base_query].keys())

    # ----- PLOT 1: Package power with A/B overlay, split by run_id -----
    fig, axes = plt.subplots(1, len(run_ids), figsize=(6 * len(run_ids), 4), sharey=True)
    if len(run_ids) == 1:
        axes = [axes]

    color_map = {'queryA': '#1f77b4', 'queryB': '#ff7f0e'}

    for ax, run_id in zip(axes, run_ids):
        vgroups = base_query_groups[base_query][run_id]
        ax.set_title(f"{base_query} — run {run_id}")

        for variant in sorted(vgroups.keys()):
            rapl_files = vgroups[variant]['rapl']
            if not rapl_files:
                continue

            all_pkg_vals = []
            all_times = []
            t_offset = 0.0

            for rapl_file in sorted(rapl_files):
                df = load_sample(rapl_file)
                if df.empty or 'timestamp_sec' not in df.columns or 'pkg_j' not in df.columns:
                    continue
                df = df[['timestamp_sec', 'pkg_j']].dropna().sort_values('timestamp_sec')
                if len(df) < 2:
                    continue

                t = df['timestamp_sec'].to_numpy(dtype=float) - df['timestamp_sec'].iloc[0]
                v = df['pkg_j'].to_numpy(dtype=float)
                all_times.append(t + t_offset)
                all_pkg_vals.append(v)
                t_offset += t[-1]

            if all_times:
                t_concat = np.concatenate(all_times)
                v_concat = np.concatenate(all_pkg_vals)
                sec = np.floor(t_concat).astype(int)
                binned_df = pd.DataFrame({'sec': sec, 'pkg': v_concat}).groupby('sec', as_index=False).mean()

                ax.plot(
                    binned_df['sec'],
                    binned_df['pkg'],
                    color=color_map.get(variant, '#333'),
                    linewidth=2,
                    label=variant
                )

        ax.set_xlabel('Elapsed Time (sec)')
        ax.set_ylabel('Mean pkg_j per 1s Bin')
        ax.legend(loc='best')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # ----- PLOT 2: Turbostat metrics with all CPUs on one plot, split by run_id -----
    fig, axes = plt.subplots(1, len(run_ids), figsize=(6 * len(run_ids), 5), sharey=True)
    if len(run_ids) == 1:
        axes = [axes]

    for ax_busy, run_id in zip(axes, run_ids):
        vgroups = base_query_groups[base_query][run_id]

        # Collect CPUs for this run_id
        all_cpus = set()
        for variant in vgroups.keys():
            for turb_file in vgroups[variant]['turb']:
                cpu_groups = parse_turbostat_file(turb_file)
                all_cpus.update(cpu_groups.keys())
        all_cpus = sorted(list(all_cpus))

        if not all_cpus:
            ax_busy.set_title(f"{base_query} — run {run_id} (no CPU data)")
            continue

        ax_mhz = ax_busy.twinx()
        ax_busy.set_title(f"{base_query} — run {run_id}")

        cpu_palette = sns.color_palette("tab10", n_colors=max(len(all_cpus), 1))
        cpu_colors = {cpu: cpu_palette[i % len(cpu_palette)] for i, cpu in enumerate(all_cpus)}
        variant_styles = {"queryA": "-", "queryB": "--"}

        for variant in sorted(vgroups.keys()):
            turb_files = vgroups[variant]['turb']
            if not turb_files:
                continue

            for cpu_num in all_cpus:
                all_busy = []
                all_mhz = []
                time_offset_sec = 0

                for turb_file in sorted(turb_files):
                    cpu_groups = parse_turbostat_file(turb_file)
                    if cpu_num not in cpu_groups:
                        continue

                    cpu_data = cpu_groups[cpu_num].copy()

                    busy_col = next((c for c in cpu_data.columns if c.lower() in ('busy%', 'busy')), None)
                    mhz_col = next((c for c in cpu_data.columns if c.lower() in ('avg_mhz', 'avgmhz')), None)
                    if busy_col is None and mhz_col is None:
                        continue

                    cpu_data['sec'] = np.arange(len(cpu_data)) + time_offset_sec
                    if busy_col:
                        all_busy.append(cpu_data.groupby('sec')[busy_col].mean())
                    if mhz_col:
                        all_mhz.append(cpu_data.groupby('sec')[mhz_col].mean())
                    time_offset_sec += len(cpu_data) + 1

                if all_busy or all_mhz:
                    color = cpu_colors.get(cpu_num, '#333')
                    style = variant_styles.get(variant, '-')
                    if all_busy:
                        combined_busy = pd.concat(all_busy).groupby(level=0).mean()
                        ax_busy.plot(
                            combined_busy.index, combined_busy.values,
                            color=color, linestyle=style, linewidth=1.2, alpha=0.9
                        )
                    if all_mhz:
                        combined_mhz = pd.concat(all_mhz).groupby(level=0).mean()
                        ax_mhz.plot(
                            combined_mhz.index, combined_mhz.values,
                            color=color, linestyle=style, linewidth=1.2, alpha=0.6
                        )

        ax_busy.set_xlabel('Elapsed Time (sec)')
        ax_busy.set_ylabel('Busy%')
        ax_mhz.set_ylabel('Avg_MHz')
        ax_busy.grid(True, alpha=0.2)

        cpu_handles = [Line2D([0], [0], color=cpu_colors[c], lw=2, label=f"CPU {c}") for c in all_cpus]
        variant_handles = [
            Line2D([0], [0], color='#333', linestyle=variant_styles[v], lw=2, label=v)
            for v in sorted(vgroups.keys()) if v in variant_styles
        ]

        legend1 = ax_busy.legend(handles=cpu_handles, title='CPU', loc='upper right',
                                 ncol=2, fontsize=8, title_fontsize=9)
        ax_busy.add_artist(legend1)
        if variant_handles:
            ax_busy.legend(handles=variant_handles, title='Variant', loc='upper left',
                           fontsize=8, title_fontsize=9)

    plt.tight_layout()
    plt.show()

    print(f"Plotted {base_query}")


In [ ]:
# Filtered SVG plots for all queries using run-id labels (PC + core count).
from pathlib import Path
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

if "parse_turbostat_file" not in globals():
    def parse_turbostat_file(path):
        """Parse turbostat CSV with whitespace separator, filter headers, extract data grouped by CPU."""
        try:
            df = pd.read_csv(path, sep=r"\s+", engine="python")
        except Exception as e:
            print(f"Failed to parse {path}: {e}")
            return {}
        if "Core" in df.columns:
            df = df[df["Core"].astype(str) != "Core"].copy()
        num_cols = ["PkgWatt", "CorWatt", "RAMWatt", "Avg_MHz", "Bzy_MHz", "Busy%", "IPC"]
        for c in num_cols:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")
        if "CPU" not in df.columns:
            return {}
        df["CPU_num"] = pd.to_numeric(df["CPU"], errors="coerce")
        df = df.dropna(subset=["CPU_num"])
        cpu_groups = {}
        for cpu_num in sorted(df["CPU_num"].unique()):
            cpu_data = df[df["CPU_num"] == cpu_num].copy()
            if not cpu_data.empty:
                cpu_groups[int(cpu_num)] = cpu_data
        return cpu_groups

if "base_query_groups" not in globals() or not base_query_groups:
    print("base_query_groups not found. Run the cell above that builds the groups first.")
else:
    save_dir = Path("plots")
    save_dir.mkdir(parents=True, exist_ok=True)

    run_id_labels = {
        "1F830D594D6929BA": "PC2_all_cores",
        "352FE3245A617222": "PC2_1_core",
        "7C61BAE5E86FAB53": "PC1_all_cores",
        "E940BCCA0B1AE2A3": "PC1_1_core",
    }
    label_to_run_ids = {}
    for rid, label in run_id_labels.items():
        label_to_run_ids.setdefault(label, []).append(rid)

    all_queries = sorted(base_query_groups.keys())
    if not all_queries:
        print("No base queries found in base_query_groups")
    else:
        color_map = {"queryA": "#1f77b4", "queryB": "#ff7f0e"}
        variant_styles = {"queryA": "-", "queryB": "--"}
        pc_layout = [
            ("PC1", "PC1_all_cores", "PC1_1_core"),
            ("PC2", "PC2_all_cores", "PC2_1_core"),
        ]

        def _collect_files(base_query, run_ids, variant, key):
            files = []
            for rid in run_ids or []:
                vgroups = base_query_groups.get(base_query, {}).get(rid, {})
                files.extend(vgroups.get(variant, {}).get(key, []))
            return sorted(files)

        def _plot_power(ax, base_query, run_ids, title_suffix):
            ax.set_title(f"{base_query} — {title_suffix}")
            max_sec = None
            if not run_ids:
                ax.text(0.5, 0.5, "No runs", ha="center", va="center")
                ax.set_xlabel("Elapsed Time (sec)")
                ax.set_ylabel("Mean pkg_j per 1s Bin")
                ax.grid(True, alpha=0.3)
                return max_sec
            for variant in ["queryA", "queryB"]:
                rapl_files = _collect_files(base_query, run_ids, variant, "rapl")
                if not rapl_files:
                    continue
                all_pkg_vals = []
                all_times = []
                t_offset = 0.0
                for rapl_file in rapl_files:
                    df = load_sample(rapl_file)
                    if df.empty or "timestamp_sec" not in df.columns or "pkg_j" not in df.columns:
                        continue
                    df = df[["timestamp_sec", "pkg_j"]].dropna().sort_values("timestamp_sec")
                    if len(df) < 2:
                        continue
                    t = df["timestamp_sec"].to_numpy(dtype=float) - df["timestamp_sec"].iloc[0]
                    v = df["pkg_j"].to_numpy(dtype=float)
                    all_times.append(t + t_offset)
                    all_pkg_vals.append(v)
                    t_offset += t[-1]
                if all_times:
                    t_concat = np.concatenate(all_times)
                    v_concat = np.concatenate(all_pkg_vals)
                    sec = np.floor(t_concat).astype(int)
                    binned_df = (
                        pd.DataFrame({"sec": sec, "pkg": v_concat})
                        .groupby("sec", as_index=False)
                        .mean()
                    )
                    if not binned_df.empty:
                        local_max = int(binned_df["sec"].max())
                        max_sec = local_max if max_sec is None else max(max_sec, local_max)
                    ax.plot(
                        binned_df["sec"],
                        binned_df["pkg"],
                        color=color_map.get(variant, "#333"),
                        linewidth=2,
                        label=variant,
                    )
            ax.set_xlabel("Elapsed Time (sec)")
            ax.set_ylabel("Mean pkg_j per 1s Bin")
            ax.grid(True, alpha=0.3)
            ax.legend(loc="best")
            return max_sec

        def _plot_busy(ax, base_query, run_ids, title_suffix):
            ax.set_title(f"{base_query} — {title_suffix}")
            max_sec = None
            if not run_ids:
                ax.text(0.5, 0.5, "No runs", ha="center", va="center")
                ax.set_xlabel("Elapsed Time (sec)")
                ax.set_ylabel("Busy%")
                ax.grid(True, alpha=0.2)
                return max_sec

            # Collect CPU list across all turbostat files in this label
            all_cpus = set()
            for variant in ["queryA", "queryB"]:
                turb_files = _collect_files(base_query, run_ids, variant, "turb")
                for turb_file in turb_files:
                    cpu_groups = parse_turbostat_file(turb_file)
                    all_cpus.update(cpu_groups.keys())
            all_cpus = sorted(all_cpus)
            if not all_cpus:
                ax.text(0.5, 0.5, "No CPU data", ha="center", va="center")
                ax.set_xlabel("Elapsed Time (sec)")
                ax.set_ylabel("Busy%")
                ax.grid(True, alpha=0.2)
                return max_sec

            cpu_palette = sns.color_palette("tab10", n_colors=max(len(all_cpus), 1))
            cpu_colors = {cpu: cpu_palette[i % len(cpu_palette)] for i, cpu in enumerate(all_cpus)}

            for variant in ["queryA", "queryB"]:
                turb_files = _collect_files(base_query, run_ids, variant, "turb")
                if not turb_files:
                    continue
                for cpu_num in all_cpus:
                    all_busy = []
                    time_offset_sec = 0
                    for turb_file in sorted(turb_files):
                        cpu_groups = parse_turbostat_file(turb_file)
                        if cpu_num not in cpu_groups:
                            continue
                        cpu_data = cpu_groups[cpu_num].copy()
                        busy_col = next((c for c in cpu_data.columns if c.lower() in ("busy%", "busy")), None)
                        if busy_col is None:
                            continue
                        cpu_data["sec"] = np.arange(len(cpu_data)) + time_offset_sec
                        all_busy.append(cpu_data.groupby("sec")[busy_col].mean())
                        time_offset_sec += len(cpu_data) + 1
                    if all_busy:
                        combined_busy = pd.concat(all_busy).groupby(level=0).mean()
                        if not combined_busy.empty:
                            local_max = int(combined_busy.index.max())
                            max_sec = local_max if max_sec is None else max(max_sec, local_max)
                        ax.plot(
                            combined_busy.index,
                            combined_busy.values,
                            color=cpu_colors.get(cpu_num, "#333"),
                            linestyle=variant_styles.get(variant, "-"),
                            linewidth=1.2,
                            alpha=0.9,
                        )
            ax.set_xlabel("Elapsed Time (sec)")
            ax.set_ylabel("Busy%")
            ax.grid(True, alpha=0.2)

            cpu_handles = [Line2D([0], [0], color=cpu_colors[c], lw=2, label=f"CPU {c}") for c in all_cpus]
            variant_handles = [
                Line2D([0], [0], color="#333", linestyle=variant_styles[v], lw=2, label=v)
                for v in ["queryA", "queryB"]
            ]
            if cpu_handles:
                legend1 = ax.legend(
                    handles=cpu_handles, title="CPU", loc="upper right",
                    ncol=2, fontsize=8, title_fontsize=9
                )
                ax.add_artist(legend1)
            if variant_handles:
                ax.legend(
                    handles=variant_handles, title="Variant", loc="upper left",
                    fontsize=8, title_fontsize=9
                )
            return max_sec

        for base_query in all_queries:
            for pc_name, label_all, label_one in pc_layout:
                fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
                max_left = _plot_power(axes[0], base_query, label_to_run_ids.get(label_all), f"{pc_name} all cores")
                max_right = _plot_power(axes[1], base_query, label_to_run_ids.get(label_one), f"{pc_name} 1 core")
                max_x = max([v for v in [max_left, max_right] if v is not None], default=None)
                if max_x is not None:
                    axes[0].set_xlim(0, max_x)
                    axes[1].set_xlim(0, max_x)
                plt.tight_layout()
                out_path = save_dir / f"single_core_filtered_power_{base_query}_{pc_name}.svg"
                fig.savefig(out_path, format="svg", bbox_inches="tight")
                print(f"Saved: {out_path}")
                plt.show()

            for pc_name, label_all, label_one in pc_layout:
                fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
                max_left = _plot_busy(axes[0], base_query, label_to_run_ids.get(label_all), f"{pc_name} all cores")
                max_right = _plot_busy(axes[1], base_query, label_to_run_ids.get(label_one), f"{pc_name} 1 core")
                max_x = max([v for v in [max_left, max_right] if v is not None], default=None)
                if max_x is not None:
                    axes[0].set_xlim(0, max_x)
                    axes[1].set_xlim(0, max_x)
                plt.tight_layout()
                out_path = save_dir / f"single_core_filtered_busy_{base_query}_{pc_name}.svg"
                fig.savefig(out_path, format="svg", bbox_inches="tight")
                print(f"Saved: {out_path}")
                plt.show()